# Mixed elasticity: H(div) stress and weak symmetry

This notebook uses the native BDM2 / discontinuous-P1 displacement / discontinuous-P1 rotation formulation. The three local rigid motions are exact. The skeleton is negative Cauchy traction; rotation approximates `(du_x/dy-du_y/dx)/2`.

The spaces and default interior/exterior traction choices follow the [2021 MHM weak-symmetry elasticity formulation](https://doi.org/10.1051/m2an/2021013). The numerical records below verify analytical data on the stated pyMHM meshes; they do not reproduce the paper's table ordinates.


The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/f7a227034dcc986ae4d7712e4f79c27142d6dff1216e52de37fd73e87bb9641b/16_mixed_elasticity-companion.zip"
COMPANION_SHA256 = "f7a227034dcc986ae4d7712e4f79c27142d6dff1216e52de37fd73e87bb9641b"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("elasticity/16_mixed_elasticity.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from pymhm.meshes.triangle import TriangleMesh



## Non-affine solenoidal patch

For `u=(x²,-2xy)`, `mu=1`, and any finite nonnegative `lambda`, the exact stress is `[[4x,-2y],[-2y,-4x]]`, force is `(-2,0)`, and rotation is `q=y`. The discrete displacement is its cellwise P1 L2 projection; its L2 error is therefore nonzero.


## User-defined weak-symmetry block equations

The primary solve declares the actual mixed coefficient fields
$(\sigma,u,q,\psi)$ through `LocalEquations`. Shared kernels integrate stress
compliance, divergence, asymmetry, physical force, rigid moments and normal
maps. The local matrix is

$$
\begin{bmatrix}
 A&\mathrm{Div}^\top&\mathrm{Asym}^\top&-S\\
 \mathrm{Div}&0&0&0\\
 \mathrm{Asym}&0&0&0\\
 -S^\top&0&0&0
\end{bmatrix}
\begin{bmatrix}\sigma\\u\\q\\\psi\end{bmatrix}
+\begin{bmatrix}0\\0\\0\\-T\end{bmatrix}\lambda
=\begin{bmatrix}0\\-f\\0\\0\end{bmatrix}.
$$

$S$ selects local boundary normal moments and $T$ maps the canonical macro
traction basis to those moments, including the incident-cell orientation.
The three exact rigid modes and their physical displacement moments are
explicit. With $B=(0,0,0,-T)^\top$, the declared global pairing is $C=-B^\top$,
so the global boundary load is the **positive** displacement moment in this
coordinate convention. `Equation` supplies that load and zero extra load in
the retained rigid coordinates. The full-displacement hydrostatic constraint
uses the independently integrated physical compliance trace identity.

The same local/global core eliminates these declared fields. Physical field
records retain stress/displacement/rotation evaluation, norms and equilibrium
checks. Solver names do not select the primary forms.

In [ ]:
import inspect
from IPython.display import Code
from examples.tutorial_stress_equations import (
    solve_bdm, mixed_equations, bdm_equations, global_equation,
)

for builder in (mixed_equations, bdm_equations, global_equation):
    display(Code(inspect.getsource(builder), language="python"))

def displacement(x: np.ndarray) -> np.ndarray:
    """Quadratic divergence-free displacement."""
    return np.column_stack((x[:, 0]**2, -2*x[:, 0]*x[:, 1]))


def stress(x: np.ndarray) -> np.ndarray:
    """Analytical Cauchy stress for mu=1."""
    value = np.zeros((len(x), 2, 2))
    value[:, 0, 0], value[:, 1, 1] = 4*x[:, 0], -4*x[:, 0]
    value[:, 0, 1] = value[:, 1, 0] = -2*x[:, 1]
    return value

system, solution = solve_bdm(
    TriangleMesh.unit_square(2), lame_lambda=1e6, lame_mu=1,
    source=(-2, 0), dirichlet=displacement,
)
assert solution.stress_l2_error(stress) < 1e-8
assert solution.rotation_l2_error(lambda x: x[:, 1]) < 2e-12
assert np.max(np.abs(solution.equilibrium_residuals())) < 3e-12
{"displacement_L2": solution.l2_error(displacement),
 "stress_L2": solution.stress_l2_error(stress),
 "force_and_moment_max": float(np.max(np.abs(solution.equilibrium_residuals())))}


## Explicit compatibility comparison

The earlier BDM solver is run separately with the identical mesh,
BDM2/P1/P1 spaces, finite Lamé ratio, body force, nonhomogeneous displacement
trace and quadrature. The declared matrix/kernel/moments and executed basis
were also checked before this notebook control. The coefficient comparison
includes stress, displacement, weak rotation, local boundary fields, trace
and all retained rigid amplitudes.

In [ ]:
from examples.formulations.application import weak_stress_elasticity as solve_elasticity_mixed

reference = solve_elasticity_mixed(
    solution.skeleton.mesh, lame_lambda=1e6, lame_mu=1,
    source=(-2, 0), dirichlet=displacement,
)
for own, previous in zip(
    (*solution.hybrid.fields, solution.hybrid.trace, *solution.hybrid.coarse),
    (*reference.hybrid.fields, reference.hybrid.trace, *reference.hybrid.coarse), strict=True,
):
    np.testing.assert_allclose(own, previous, atol=1e-12, rtol=1e-10)
np.testing.assert_allclose(solution.l2_error(displacement), reference.l2_error(displacement), atol=1e-12, rtol=1e-10)
np.testing.assert_allclose(solution.stress_l2_error(stress), reference.stress_l2_error(stress), atol=1e-12, rtol=1e-10)
np.testing.assert_allclose(solution.rotation_l2_error(lambda x: x[:, 1]),
                           reference.rotation_l2_error(lambda x: x[:, 1]), atol=1e-12, rtol=1e-10)
print("Original BDM stress/displacement/rotation and rigid coefficients agree within atol=1e-12 and rtol=1e-10")

## Five meshes, seven Lamé ratios, and physical moments

The archived analytical study uses diagonal unit-square macrotriangles, local refinement two, interior P1 traction, and exterior P2 on each fine boundary edge. Errors are integrated with order-10 quadrature. Weak symmetry refers to P1 moments; the raw tensor need not be symmetric pointwise. The native integration suite separately compares all fields with independently assembled DOLFINx/UFL and Basix BDM2/P1/P1.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    records = json.loads((ROOT / "examples/results/mixed-elasticity.json").read_text())
    records["convergence"]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity/convergence-and-incompressibility.png")))
    display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity/smooth-fields.png")))


## Oscillatory modulus analytical case

The 2021 data use `E=100[1+0.3 sin(10π(x−0.5))cos(10πy)]`, `nu=0.3`, and displacement `(x²y² cos(6πx)sin(7πy)/27, exp(y)sin(4πx)/5)`. The prescribed boundary data are the actual nonhomogeneous trace. Force includes the gradients of E and is checked independently by complex-step stress differentiation in the public analytical gallery example. These results use 512 macrotriangles and are not an assertion that Table 3 has been reproduced.


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    records["oscillatory_2021_analytical_data"]


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity/oscillatory-fields.png")))


## Limits

The finite-coefficient sweep reaches `lambda/mu=1e6` (`nu≈0.4999995`). At much larger ratios, floating-point error in mean hydrostatic stress can grow; infinite lambda is not supported. Uniform behavior for this stable family does not establish robustness for arbitrary coefficient contrast or arbitrary skeleton choices. Stress divergence errors and integrated equilibrium defects measure different quantities.


## Independent assembly of the published oscillatory case

DOLFINx/UFL/Basix assembles the same variable modulus, source and nonzero displacement trace. A native normal-moment restriction preserves the declared macro traction space, including the enriched local families. The compared quantities are physical stress, displacement and independent rotation. Native code agreement and analytical convergence do not resolve the article's unreported mesh convention; the historical table ordinates remain distinct.


In [ ]:
if HISTORICAL_REPLAY:
    native = json.loads((ROOT / "examples/results/mixed-elasticity/native-discrete-verification.json").read_text())
    for row in native["rows"]:
        norms = row["comparisons"][-1]
        print(row["case"], {field: norms[field]["relative_difference"] for field in ("stress", "displacement", "rotation")})
    display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity/native-displacement-rotation.png")))
    display(Image(filename=str(ROOT / "docs/figures/mixed-elasticity/native-stress.png")))
